# Data cleaning

## Load data

In [ ]:
# Load CSV file
import pandas as pd
import numpy as np
import scipy as sp
import pingouin as pg

df = pd.read_csv("45849-s3-sf_TEXT.csv")

# Remove question-text row and ImportId row
df = df.iloc[2:].copy()

# Create working dataset
data = df.copy()

print("Starting sample:", len(data))

Starting sample: 713


## Check variables

In [ ]:
print(data.columns.tolist())

['StartDate', 'EndDate', 'Status', 'IPAddress', 'Progress', 'Duration (in seconds)', 'Finished', 'RecordedDate', 'ResponseId', 'RecipientLastName', 'RecipientFirstName', 'RecipientEmail', 'ExternalReference', 'LocationLatitude', 'LocationLongitude', 'DistributionChannel', 'UserLanguage', 'Q_RecaptchaScore', 'Age', 'Founder', 'Gender', 'Gender_4_TEXT', 'Education', 'Origin', 'Residence', 'Founder-Status', 'Founder Stage', 'Time', 'Months', 'Years', 'Industry', 'Industry.1', 'Projects', 'Uncertainty', 'Mini-IPIP 1', 'Mini-IPIP 2', 'Mini-IPIP 3', 'Mini-IPIP 4', 'IPIP 120 1', 'IPIP 120 2', 'IPIP 120 3', 'IPIP 120 4', 'CPM 1', 'CPM 2', 'CPM 3', 'CPM 4', 'CPM 5', 'CPM 6', 'CPM 7', 'CPM 8', 'CPM 9', 'CPM 10', 'CPM 11', 'CPM 12', 'CPM 13', 'CPM 14', 'CPM 15', 'CPM 16', 'CPM 17', 'CPM 18', 'CPM 19', 'CPM 20', 'CPM 21', 'CPM 22', 'CPM 23', 'CPM 24', 'CPM 25', 'CPM 26', 'CPM 27', 'APS 1', 'APS 2', 'APS 3', 'APS 4', 'APS 5', 'APS 6', 'APS 7', 'APS 8', 'APS 9', 'APS 10', 'APS 11', 'APS 12', 'APS 13

### Remove irrelevant variables

In [ ]:
data = data.drop(columns=[
    'StartDate', 'EndDate', 'Status', 'IPAddress', 'Progress', 'Duration (in seconds)', 'RecordedDate', 'ResponseId', 'RecipientLastName', 'RecipientFirstName', 'RecipientEmail', 
    'ExternalReference', 'LocationLatitude', 'LocationLongitude', 'DistributionChannel', 'UserLanguage', 'Q_RecaptchaScore', 'Gender_4_TEXT', 'Founder Stage', 'Projects', 
    'Uncertainty', 'Mini-IPIP 1', 'Mini-IPIP 2', 'Mini-IPIP 3', 'Mini-IPIP 4', 'IPIP 120 1', 'IPIP 120 2', 'IPIP 120 3', 'IPIP 120 4', 'APS 1', 'APS 2', 'APS 3', 'APS 4', 
    'APS 5', 'APS 6', 'APS 7', 'APS 8', 'APS 9', 'APS 10', 'APS 11', 'APS 12', 'APS 13', 'APS 14', 'APS 15', 'APS 16', 'APS 17', 'APS 18', 'APS 19', 'APS 20', 'clicked', 
    'norms', 'project', 'source', 'results', 'Q_DataPolicyViolations', 'CPM 2', 'CPM 3', 'CPM 5', 'CPM 6', 'CPM 8', 'CPM 9', 'CPM 11', 'CPM 12', 'CPM 14', 'CPM 15', 'CPM 17', 
    'CPM 18', 'CPM 20', 'CPM 21', 'CPM 23', 'CPM 24', 'CPM 26', 'CPM 27'
])
print(data.columns.tolist())

['Finished', 'Age', 'Founder', 'Gender', 'Education', 'Origin', 'Residence', 'Founder-Status', 'Time', 'Months', 'Years', 'Industry', 'Industry.1', 'CPM 1', 'CPM 4', 'CPM 7', 'CPM 10', 'CPM 13', 'CPM 16', 'CPM 19', 'CPM 22', 'CPM 25', 'PtP 1', 'PtP 2', 'PtP 3', 'PtP 4', 'PtP 5', 'PtP 6', 'PtP 7', 'PtP 8', 'PtP 9', 'PtP 10', 'PtP 11', 'PtP 12', 'PtP 13', 'PtP 14', 'PtP 15', 'PtP 16', 'PtP 17', 'PtP 18', 'PtP 19', 'PtP 20', 'PtP 21', 'PtP 22', 'PtP 23', 'PtP 24', 'PtP 25', 'PtP 26', 'PtP 27', 'PtP 28', 'PtP 29', 'PtP 30', 'PtP 31', 'PtP 32', 'PtP 33', 'PtP 34', 'PtP 35', 'PtP 36', 'PtP 37', 'PtP 38', 'PtP 39', 'PtP 40', 'PtP 41', 'PtP 42', 'PtP 43', 'PtP 44', 'PtP 45', 'PtP 46', 'PtP 47', 'PtP 48', 'PtP 49', 'PtP 50', 'PtP 51', 'PtP 52', 'PtP 53', 'PtP 54', 'PtP 55', 'PtP 56', 'PtP 57', 'PtP 58', 'PtP 59', 'PtP 60', 'PtP 61', 'PtP 62', 'PtP 63', 'PtP 64']


### Change text to numbers

In [ ]:
# CPM questionnaire items
cpm_items = [
    "CPM 1",
    "CPM 4",
    "CPM 7",
    "CPM 10",
    "CPM 13",
    "CPM 16",
    "CPM 19",
    "CPM 22",
    "CPM 25"
]

# PtP questionnaire items
ptp_original = [f"PtP {i}" for i in range(1, 65)]


In [ ]:
cpm_mapping = {
    "Strongly disagree": 1,
    "Disagree": 2,
    "Neither agree nor disagree": 3,
    "Agree": 4,
    "Strongly agree": 5
}
data[cpm_items] = (
    data[cpm_items]
    .replace(cpm_mapping)
)

data[cpm_items] = data[cpm_items].astype("Int64")

/var/folders/sk/vsp51gvs6d91trj60vbpz9hr0000gn/T/ipykernel_83774/1069375689.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(cpm_mapping)


In [ ]:
ptp_mapping = {
    "Never or hardly ever": 1,
    "Rarely": 2,
    "Sometimes": 3,
    "About half the time": 4,
    "Often": 5,
    "Very often": 6,
    "Always or nearly always": 7
}
data[ptp_original] = (
    data[ptp_original]
    .replace(ptp_mapping)
)

data[ptp_original] = data[ptp_original].astype("Int64")

In [ ]:
print(data[cpm_items].head())
print(data[ptp_original].head())

In [ ]:
# Check text vs numbers
print(data.dtypes)

In [ ]:
# Columns that should contain whole numbers
numeric_columns = [
    "Age",
    "Time",
    "Months",
    "Years",
]

# Convert columns to numeric first
for col in numeric_columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")

# Convert to whole-number format while allowing missing values
data[numeric_columns] = data[numeric_columns].astype("Int64")

In [ ]:
print(data.dtypes)

### Check data

In [ ]:
data[cpm_items + ptp_original].head()

### Check data again

In [ ]:
# Count answered questions for each participant
data["CPM_completed"] = data[cpm_items].notna().sum(axis=1)
data["PtP_completed"] = data[ptp_original].notna().sum(axis=1)

print("CPM: at least 1 question answered =", (data["CPM_completed"] > 0).sum())
print("PtP: at least 1 question answered =", (data["PtP_completed"] > 0).sum())

## Exclusion criteria

In [ ]:
# Check data
print(data.head())
print(data.shape)

# Check age
print("Missing age:")
print(data["Age"].isna().sum())

print("\nAge distribution:")
print(data["Age"].describe())

# Check founder responses
print("\nFounder responses:")
print(data["Founder"].value_counts(dropna=False))

### Apply exclusion criteria (age, founder)

In [ ]:
# 1. Remove missing age
data = data[data["Age"].notna()]
print("After missing age removed:", len(data))

# 2. Remove participants under 18
data = data[data["Age"] >= 18]
print("After under 18 removed:", len(data))

# 3. Remove missing founder response
data = data[data["Founder"].notna()]
print("After missing founder response removed:", len(data))

# 4. Remove non-founders
data = data[data["Founder"] != "No"]
print("After non-founders removed:", len(data))


In [ ]:
# Count completed items
data["CPM_completed"] = data[cpm_items].notna().sum(axis=1)
data["PtP_completed"] = data[ptp_original].notna().sum(axis=1)


# Remove participants who didn't start either inventory
didnt_start_either = (
    (data["CPM_completed"] == 0) &
    (data["PtP_completed"] == 0)
)

print(
    "Didn't start either inventory:",
    didnt_start_either.sum()
)

data = data[~didnt_start_either].copy()

print(
    "After removing participants who didn't start either inventory:",
    len(data)
)


# Remove participants who completed only one inventory
only_one_inventory = (
    ((data["CPM_completed"] > 0) & (data["PtP_completed"] == 0)) |
    ((data["CPM_completed"] == 0) & (data["PtP_completed"] > 0))
)

print(
    "Only completed one inventory:",
    only_one_inventory.sum()
)

data = data[~only_one_inventory].copy()

print("Final sample:", len(data))

## Reverse scoring

In [ ]:
# Select all 64 PtP items from the main dataset
ptp_original = data[[f"PtP {i}" for i in range(1, 65)]].copy()

print(type(ptp_original))

In [ ]:
# Define items that need reverse scoring
reverse_ptp_original = [
    "PtP 5", "PtP 6", "PtP 7", "PtP 8",
    "PtP 11",
    "PtP 13", "PtP 14", "PtP 15", "PtP 16", "PtP 17",
    "PtP 18", "PtP 19",
    "PtP 24", "PtP 25",
    "PtP 30", "PtP 31", "PtP 32",
    "PtP 34",
    "PtP 37",
    "PtP 39", "PtP 40", "PtP 41", "PtP 42", "PtP 43", "PtP 44",
    "PtP 48", "PtP 49", "PtP 50", "PtP 51", "PtP 52",
    "PtP 56", "PtP 57", "PtP 58",
    "PtP 61"
]
# Make a copy of the original PtP data
ptp_items = ptp_original.copy()

# Reverse score the selected items
ptp_items[reverse_ptp_original] = 8 - ptp_original[reverse_ptp_original]

In [ ]:
print(ptp_items.head())

## Creating the moderator variable

In [ ]:
# Define the R&D classifications
industry_classification = {
    "Agriculture": "lRD",
    "Forestry": "lRD",
    "Mining": "lRD",
    "Energy and water supply": "lRD",
    "Waste/Recycling": "lRD",
    "Construction": "lRD",
    "Trades e.g. Plumbing/Electrical": "lRD",
    "Manufacturing": "hRD",
    "Wholesale trade": "lRD",
    "Retail trade": "lRD",
    "Real estate": "lRD",
    "Hospitality": "lRD",
    "Delivery services": "lRD",
    "Transport/Logistics": "lRD",
    "Cleaning/Maintenance": "lRD",
    "Technical services": "mRD",
    "Travel/Tourism": "lRD",
    "Entertainment": "lRD",
    "Recreation": "lRD",
    "Arts": "lRD",
    "Education/Training": "mRD",
    "Research/Academia": "hRD",
    "Health care": "mRD",
    "Social services": "mRD",
    "Administration": "lRD",
    "IT/Telecommunication": "mRD",
    "Finance/Insurance": "lRD",
    "Environment/Sustainability": "lRD"
}

In [ ]:
# Split multiple industries for each participant
data["Industry_list"] = data["Industry.1"].str.split(",")

# Classify each selected industry
data["RD_categories"] = data["Industry_list"].apply(
    lambda industries: [
        industry_classification[industry.strip()]
        for industry in industries
        if industry.strip() in industry_classification
    ] if isinstance(industries, list) else []
)

# Count number of lRD, mRD, hRD industries selected
data["lRD_count"] = data["RD_categories"].apply(lambda x: x.count("lRD"))
data["mRD_count"] = data["RD_categories"].apply(lambda x: x.count("mRD"))
data["hRD_count"] = data["RD_categories"].apply(lambda x: x.count("hRD"))


In [ ]:
# Calculate weighted R&D intensity score
data["RDweight"] = (
    (data["lRD_count"] * 1) +
    (data["mRD_count"] * 2) +
    (data["hRD_count"] * 3)
) / (
    data["lRD_count"] +
    data["mRD_count"] +
    data["hRD_count"]
)


# Create final moderator variable
# RDweight < 2 = lRD
# RDweight >= 2 = hRD

data["RD_moderator"] = data["RDweight"].apply(
    lambda x: pd.NA if pd.isna(x)
    else "lRD" if x < 2.00
    else "hRD"
)


In [ ]:
# Check final groups
print(data["RD_moderator"].value_counts(dropna=False))

## Demographic information

In [ ]:
# Variables to summarise (excluding Industry.1)
demographic_vars = [
    "Gender",
    "Education",
    "Origin",
    "Residence",
    "Founder-Status",
    "Months",
    "Years"
]

# Generate frequency tables for normal variables
for var in demographic_vars:
    print("\n====================")
    print(var)
    print("====================")
    
    freq = data[var].value_counts(dropna=False)
    percent = data[var].value_counts(normalize=True, dropna=False) * 100
    
    summary = pd.DataFrame({
        "Frequency (n)": freq,
        "Percentage (%)": percent.round(2)
    })
    
    print(summary)


# Separate analysis for Industry.1 (multiple responses)
print("\n====================")
print("Industry")
print("====================")

industry_freq = (
    data["Industry.1"]
    .dropna()
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
)

industry_percent = (industry_freq / len(data)) * 100

industry_summary = pd.DataFrame({
    "Frequency (n)": industry_freq,
    "Percentage (%)": industry_percent.round(2)
})

print(industry_summary)

## Frequency check for variables

In [ ]:
# CPM frequency table including total responses

cpm_frequency_summary = pd.DataFrame()

for item in cpm_items:
    freq = data[item].value_counts(dropna=False).sort_index()
    
    # Add total responses (excluding NaN)
    freq["Total responses"] = data[item].notna().sum()
    
    cpm_frequency_summary[item] = freq

print(cpm_frequency_summary)

In [ ]:
# PtP frequency table including total responses

ptp_frequency_summary = pd.DataFrame()

for item in ptp_items:
    freq = data[item].value_counts(dropna=False).sort_index()
    
    # Add total responses (excluding NaN)
    freq["Total responses"] = data[item].notna().sum()
    
    ptp_frequency_summary[item] = freq

print(ptp_frequency_summary)

In [ ]:
# RD moderator frequency table

rd_frequency = data["RD_moderator"].value_counts(dropna=False)

rd_summary = pd.DataFrame({
    "Frequency (n)": rd_frequency,
    "Percentage (%)": (rd_frequency / len(data) * 100).round(2)
})

print(rd_summary)

## Missingness check

In [ ]:
# Count item-level missing responses

cpm_missing = data[cpm_items].isna().sum().sum()
ptp_missing = data[ptp_items].isna().sum().sum()
rd_missing = data["RD_moderator"].isna().sum()

# Total possible responses
cpm_total_possible = data[cpm_items].size
ptp_total_possible = data[ptp_items].size
rd_total_possible = len(data)

# Create summary table
missing_summary = pd.DataFrame({
    "Variable": [
        "CPM",
        "PtP",
        "RD moderator",
        "Total"
    ],
    "Missing responses (n)": [
        cpm_missing,
        ptp_missing,
        rd_missing,
        cpm_missing + ptp_missing + rd_missing
    ],
    "Possible responses (n)": [
        cpm_total_possible,
        ptp_total_possible,
        rd_total_possible,
        cpm_total_possible + ptp_total_possible + rd_total_possible
    ],
    "Missing (%)": [
        round(cpm_missing / cpm_total_possible * 100, 2),
        round(ptp_missing / ptp_total_possible * 100, 2),
        round(rd_missing / rd_total_possible * 100, 2),
        round((cpm_missing + ptp_missing + rd_missing) /
              (cpm_total_possible + ptp_total_possible + rd_total_possible) * 100, 2)
    ]
})

print(missing_summary)

In [ ]:
# Missingness
mcar_data.isna().sum().sum()

In [ ]:
# Show variables with missing values
missing_by_variable = mcar_data.isna().sum()

print(missing_by_variable[missing_by_variable > 0])

In [ ]:
# Show participants with missing responses
missing_by_person = mcar_data.isna().sum(axis=1)

print(missing_by_person[missing_by_person > 0])

In [ ]:
# View the missing items for participant 30
data.loc[30, mcar_data.columns][data.loc[30, mcar_data.columns].isna()]

In [ ]:
data.loc[168, mcar_data.columns][data.loc[168, mcar_data.columns].isna()]

In [ ]:
data.loc[363, mcar_data.columns][data.loc[363, mcar_data.columns].isna()]

In [ ]:
missing_by_item = mcar_data.isna().sum().sort_values(ascending=False)

print(missing_by_item[missing_by_item > 0])

## Descriptive Statistics

In [ ]:
# Calculate CPM average score per participant
data["CPM_total"] = data[cpm_items].mean(axis=1)

# Calculate PtP average score per participant
data["PtP_total"] = data[ptp_items].mean(axis=1)

#Descriptives
inventory_table = data[["CPM_total", "PtP_total"]].agg(
    ["count", "mean", "std", "min", "max"]
).T.round(2)

print(inventory_table)

## Assumption Testing

### Centring and interaction terms

In [ ]:
normality_vars = [
    "CPM_total",
    "PtP_total"
]

In [ ]:
# Skewness
from scipy.stats import skew

for var in normality_vars:
    values = data[var].dropna()
    
    print(
        var,
        "Skewness:",
        round(skew(values), 3)
    )

 Skewness is between -1 and 1, indicating univariate normality

In [ ]:
# Z Skewness
import numpy as np

# can turn this into function
for var in normality_vars:
    
    values = data[var].dropna()
    
    skewness = skew(values)
    
    standard_error = np.sqrt(6 / len(values))
    
    z_skew = skewness / standard_error
    
    print(
        var,
        "Z-skew:",
        round(z_skew, 3)
    )

In [ ]:
# Shapiro Wilk
from scipy.stats import shapiro

for var in normality_vars:
    
    values = data[var].dropna()
    
    stat, p = shapiro(values)
    
    print(
        var,
        "Shapiro-Wilk W:",
        round(stat,3),
        "p:",
        round(p,3)
    )